# 🔬 NeoScan AI — Notebook 2: CNN Phase 1 (Binary Classifier)
**Project:** NeoScan AI — Neonatal LUS RDS Triage | SRHU
**Notebook:** 2 of 3 — Image Pipeline + MobileNetV3-Small Training

---
### ✅ FIRST TIME: Runtime → Change runtime type → T4 GPU → then Run All
### 🔁 RETURNING AFTER TRAINING: Run Cells 1, 2, 7 (define model), 9 (HF login), then Cell 10b (load weights)

---
Pipeline:
Dataset download → Frame extraction (SSIM dedup) → Synthetic neonatal transforms
→ Phase 1 Binary Training → HuggingFace checkpoint save

Expected time on T4 GPU: ~35-45 min

In [ ]:
!pip install -q gradio scikit-learn pandas numpy torch torchvision opencv-python-headless scikit-image pillow tqdm huggingface_hub

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Download dataset
!git clone --depth=1 https://github.com/jannisborn/covid19_ultrasound.git
import os
print(os.listdir("covid19_ultrasound/data"))

In [ ]:
# Frame extraction with SSIM deduplication
import os, json, random
import numpy as np
from pathlib import Path
from skimage.metrics import structural_similarity as ssim
import cv2

random.seed(42)
np.random.seed(42)
rng = np.random.default_rng(42)

os.makedirs("data/frames/Normal",   exist_ok=True)
os.makedirs("data/frames/Abnormal", exist_ok=True)

FRAMES_ROOT = Path("covid19_ultrasound/data/pocus_data/frames")
SSIM_THRESH = 0.98
ABNORMAL_PREFIXES = ("Cov", "Pneu", "Vir")

label_map = {}
saved_normal = saved_abnormal = 0

for folder in sorted(FRAMES_ROOT.rglob("*")):
    if not folder.is_dir(): continue
    label = 1 if folder.name.startswith(ABNORMAL_PREFIXES) else 0
    label_str = "Abnormal" if label else "Normal"
    img_files = sorted([p for p in folder.iterdir()
                        if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".bmp")])
    prev_gray = None
    for img_path in img_files:
        try:
            img = cv2.imread(str(img_path))
            if img is None: continue
            img_resized = cv2.resize(img, (224, 224))
            gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
            if prev_gray is not None:
                score, _ = ssim(prev_gray, gray, full=True)
                if score > SSIM_THRESH: continue
            prev_gray = gray
            out_name = f"{folder.name}_{img_path.stem}.jpg"
            out_path = f"data/frames/{label_str}/{out_name}"
            cv2.imwrite(out_path, img_resized)
            label_map[out_path] = label
            if label == 0: saved_normal += 1
            else:          saved_abnormal += 1
        except Exception as e:
            print(f"Error {img_path}: {e}")

with open("data/label_map.json", "w") as f:
    json.dump(label_map, f)

print(f"Total frames saved: {saved_normal + saved_abnormal}")
print(f"  Normal:   {saved_normal}")
print(f"  Abnormal: {saved_abnormal}")
print("Saved label map to: data/label_map.json")

In [ ]:
# Apply 6 synthetic neonatal domain adaptation transforms
# For each frame, creates ONE transformed copy in data/synthetic/
import os, random
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm

random.seed(42)
np.random.seed(42)

os.makedirs("data/synthetic/Normal",   exist_ok=True)
os.makedirs("data/synthetic/Abnormal", exist_ok=True)

def transform_1_brightness_boost(img):
    """Multiply pixel values by 1.15, clip to 0-255."""
    return np.clip(img.astype(np.float32) * 1.15, 0, 255).astype(np.uint8)

def transform_2_contrast_reduction(img):
    """Blend toward mean: output = img*0.8 + mean*0.2."""
    return np.clip(img.astype(np.float32) * 0.8 + img.mean() * 0.2, 0, 255).astype(np.uint8)

def transform_3_speckle_noise(img):
    """Add Gaussian noise (mean=0, std=12), clip to 0-255."""
    return np.clip(img.astype(np.float32) + np.random.normal(0, 12, img.shape), 0, 255).astype(np.uint8)

def transform_4_narrow_field_crop(img):
    """Crop center 80% of width and height, resize back to 224x224."""
    h, w = img.shape[:2]
    return cv2.resize(img[int(h*0.10):int(h*0.90), int(w*0.10):int(w*0.90)], (224, 224))

def transform_5_rib_shadow(img):
    """Darken horizontal bands at 30% and 70% of image height (band width=8px, multiply by 0.6)."""
    result = img.astype(np.float32)
    h = img.shape[0]
    for cf in [0.30, 0.70]:
        c = int(h * cf)
        result[max(0, c-4):min(h, c+4), :] *= 0.6
    return np.clip(result, 0, 255).astype(np.uint8)

def transform_6_near_field_bright(img):
    """Top 25% of image multiplied by 1.2, clip to 255."""
    result = img.astype(np.float32)
    result[:int(img.shape[0] * 0.25), :] *= 1.2
    return np.clip(result, 0, 255).astype(np.uint8)

TRANSFORMS = [
    transform_1_brightness_boost, transform_2_contrast_reduction,
    transform_3_speckle_noise,    transform_4_narrow_field_crop,
    transform_5_rib_shadow,       transform_6_near_field_bright,
]

total_synthetic = 0
for label_str in ["Normal", "Abnormal"]:
    src_dir = Path(f"data/frames/{label_str}")
    dst_dir = Path(f"data/synthetic/{label_str}")
    frames = sorted(src_dir.glob("*.jpg"))
    for i, fp in enumerate(tqdm(frames, desc=label_str)):
        img = cv2.imread(str(fp))
        if img is None: continue
        cv2.imwrite(str(dst_dir / fp.name), TRANSFORMS[i % len(TRANSFORMS)](img))
        total_synthetic += 1

print(f"Total synthetic frames created: {total_synthetic}")

In [ ]:
# Patient-level 70/15/15 split
# Original frames only → stratified 70/15/15
# ALL synthetic frames added to TRAIN set only (no data leakage to val/test)
import os, json, random
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from collections import Counter

random.seed(42)
np.random.seed(42)
os.makedirs("data", exist_ok=True)

orig_records = []
for label_int, label_str in [(0, "Normal"), (1, "Abnormal")]:
    for fp in sorted(Path(f"data/frames/{label_str}").glob("*.jpg")):
        orig_records.append({"path": str(fp), "label": label_int})

labels_arr = [r["label"] for r in orig_records]
idx = list(range(len(orig_records)))
idx_train, idx_temp, _, y_temp = train_test_split(
    idx, labels_arr, test_size=0.30, stratify=labels_arr, random_state=42)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, stratify=y_temp, random_state=42)

train_orig   = [orig_records[i] for i in idx_train]
val_records  = [orig_records[i] for i in idx_val]
test_records = [orig_records[i] for i in idx_test]

synth_records = []
for label_int, label_str in [(0, "Normal"), (1, "Abnormal")]:
    for fp in sorted(Path(f"data/synthetic/{label_str}").glob("*.jpg")):
        synth_records.append({"path": str(fp), "label": label_int})

train_records = train_orig + synth_records
splits = {"train": train_records, "val": val_records, "test": test_records}
with open("data/image_splits.json", "w") as f:
    json.dump(splits, f)

label_names = {0: "Normal", 1: "Abnormal"}
for split_name, recs in splits.items():
    c = Counter(r["label"] for r in recs)
    print(f"{split_name} ({len(recs)} frames): ",
          ", ".join(f"{label_names[l]}: {v}" for l, v in sorted(c.items())))
print("\nSaved to: data/image_splits.json")

In [ ]:
# Define Phase1Model and Phase1Dataset
import random
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


class Phase1Model(nn.Module):
    def __init__(self):
        super().__init__()
        from torchvision.models import mobilenet_v3_small
        base = mobilenet_v3_small(weights=None)
        base.classifier[3] = nn.Linear(base.classifier[3].in_features, 2)
        self.model = base
    def forward(self, x):
        return self.model(x)


MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

TRAIN_TRANSFORM = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class Phase1Dataset(Dataset):
    def __init__(self, records, split="train"):
        self.records   = records
        self.transform = TRAIN_TRANSFORM if split == "train" else EVAL_TRANSFORM
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        r = self.records[idx]
        img = Image.open(r["path"]).convert("RGB")
        return self.transform(img), int(r["label"])


print("Phase1Model and Phase1Dataset defined.")
print(f"Parameters: {sum(p.numel() for p in Phase1Model().parameters()):,}")

In [ ]:
# Training + test evaluation
import os, json, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from collections import Counter
from sklearn.metrics import f1_score, classification_report, confusion_matrix

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
os.makedirs("models/phase1", exist_ok=True)

with open("data/image_splits.json") as f:
    splits = json.load(f)

train_ds = Phase1Dataset(splits["train"], split="train")
val_ds   = Phase1Dataset(splits["val"],   split="val")
test_ds  = Phase1Dataset(splits["test"],  split="test")

train_labels = [r["label"] for r in splits["train"]]
counts = Counter(train_labels)
sampler = WeightedRandomSampler(
    [1.0 / counts[l] for l in train_labels], len(train_labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,    num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False,    num_workers=2)

model = Phase1Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 2.0]).to(device))

EPOCHS, PATIENCE = 20, 5
best_abnormal_f1 = -1.0
patience_counter = 0

print(f"{'Epoch':>5} {'TrainLoss':>10} {'ValLoss':>10} {'ValAcc':>8} {'AbnF1':>8}")
print("-" * 47)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(imgs)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            val_loss += criterion(logits, labels).item() * len(imgs)
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    val_loss /= len(val_ds)
    val_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    abnormal_f1 = f1_score(all_labels, all_preds, pos_label=1, average="binary", zero_division=0)
    scheduler.step()
    print(f"{epoch:>5} {train_loss:>10.4f} {val_loss:>10.4f} {val_acc:>8.4f} {abnormal_f1:>8.4f}")

    if abnormal_f1 > best_abnormal_f1:
        best_abnormal_f1 = abnormal_f1
        patience_counter = 0
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "best_abnormal_f1": best_abnormal_f1},
                   "models/phase1/best_phase1.pth")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest val Abnormal-F1: {best_abnormal_f1:.4f}")

# --- Test set evaluation ---
print("\n" + "=" * 55)
print("TEST SET EVALUATION")
print("=" * 55)

ckpt = torch.load("models/phase1/best_phase1.pth", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        test_preds.extend(model(imgs.to(device)).argmax(1).cpu().tolist())
        test_labels.extend(labels.tolist())

target_names = ["Normal", "Abnormal"]
print(classification_report(test_labels, test_preds, target_names=target_names, digits=4))
acc = sum(p == l for p, l in zip(test_preds, test_labels)) / len(test_labels)
print(f"Accuracy:     {acc:.4f}")
print(f"Normal-F1:    {f1_score(test_labels, test_preds, pos_label=0, average='binary', zero_division=0):.4f}")
print(f"Abnormal-F1:  {f1_score(test_labels, test_preds, pos_label=1, average='binary', zero_division=0):.4f}")
print(f"Macro-F1:     {f1_score(test_labels, test_preds, average='macro', zero_division=0):.4f}")

cm = confusion_matrix(test_labels, test_preds)
print("\nConfusion Matrix:")
print(f"{'':12}{'Normal':>10}{'Abnormal':>12}")
for i, row in enumerate(cm):
    print(f"{target_names[i]:12}" + "".join(f"{v:>10}" for v in row))

In [ ]:
# --- HUGGINGFACE CHECKPOINT SAVE ---
# Setup: Colab left panel → key icon → Add secret: Name=HF_TOKEN, Value=your_write_token
import json, os
from huggingface_hub import HfApi, login
from google.colab import userdata

token = userdata.get('HF_TOKEN')
login(token=token)

api = HfApi()
repo_id = "YOUR_HF_USERNAME/neoscan-phase1"  # <-- CHANGE THIS

api.create_repo(repo_id=repo_id, exist_ok=True)
api.upload_file(
    path_or_fileobj="models/phase1/best_phase1.pth",
    path_in_repo="best_phase1.pth",
    repo_id=repo_id
)

import json
config = {
    "model": "MobileNetV3-Small",
    "num_classes": 2,
    "classes": ["Normal", "Abnormal"],
    "input_size": [3, 224, 224],
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std":  [0.229, 0.224, 0.225]
}
with open("models/phase1/phase1_config.json","w") as f:
    json.dump(config, f, indent=2)
api.upload_file(
    path_or_fileobj="models/phase1/phase1_config.json",
    path_in_repo="phase1_config.json",
    repo_id=repo_id
)
print(f"✅ Saved to: https://huggingface.co/{repo_id}")

## 🔁 RETURNING AFTER TRAINING?

Run ONLY:
- Cell 1 — Install (always needed after Colab resets)
- Cell 2 — GPU check
- Cell 7 — Define Phase1Model class (must define before loading weights)
- Cell 9 — HF login
- Cell 10b — Load from HuggingFace

SKIP: Cells 3, 4, 5, 6, 8 (dataset + training already done)

In [ ]:
# Load Phase 1 weights from HuggingFace (run when returning)
from huggingface_hub import hf_hub_download

path = hf_hub_download(repo_id="YOUR_HF_USERNAME/neoscan-phase1", filename="best_phase1.pth")
phase1_model = Phase1Model().to(device)
phase1_model.load_state_dict(torch.load(path, map_location=device))
phase1_model.eval()
print("✅ Phase 1 model loaded from HuggingFace")

In [ ]:
# Backup: download .pth directly to your PC
from google.colab import files
files.download("models/phase1/best_phase1.pth")